# Chapter 6 — Abstract Linear (Vector) Spaces: SageMath Companion

Symbolic, exact-arithmetic counterpart to `code/python/06_abstract_spaces.ipynb`. We use Sage's `PolynomialRing(QQ)` for polynomials, `Matrix(QQ, ...)` for matrices, and `Matrix(SR, ...)` / `var(...)` for symbolic axiom verification. Every hand calculation in `notes.md` and `worked-examples.md` can be cross-checked exactly here.

**To run this notebook:** open in [CoCalc](https://cocalc.com/) (no install needed), or locally with the SageMath kernel (`sage -n jupyter`).

**Kernel:** SageMath.

**Layout:**

1. The eight axioms — symbolic verification on `P₂`
2. Polynomials in Sage — `PolynomialRing(QQ)` and the coefficient bridge
3. Linear independence in `P₃` via coordinates + rank
4. Non-standard coordinates in `P₂` — including the **Lagrange basis**
5. Matrix of differentiation `D : P₃ → P₂` — symbolic
6. Composition `D ∘ M_x` as matrix product
7. Matrix of `T(p) = p + p'` on `P₂` and the **similarity formula** `[T]_{𝓑'} = P [T]_𝓑 P⁻¹`
8. Isomorphisms — `P₃ ≅ ℝ⁴` and `M_{2×2} ≅ ℝ⁴`
9. Subspace of trace-zero `2×2` matrices — basis & dimension
10. **Exercises** with solutions


## 1. The eight axioms — symbolic verification on `P₂`

We verify each axiom of a vector space for `P₂` (polynomials of degree ≤ 2) using Sage's symbolic ring `SR`. Generic elements `p, q, r` are formed with symbolic coefficients and the axioms reduce to identities of polynomials.

In [ ]:
# Symbolic coefficients. Use `t` for the indeterminate so it does not collide
# with Sage's polynomial ring later in the notebook.
a0, a1, a2, b0, b1, b2, c0, c1, c2, c, d = var('a0 a1 a2 b0 b1 b2 c0 c1 c2 c d')
t = var('t')

p = a0 + a1*t + a2*t^2
q = b0 + b1*t + b2*t^2
r = c0 + c1*t + c2*t^2

# Axiom 1: associativity
ax1 = ((p + q) + r - (p + (q + r))).expand()
print('Axiom 1 (associativity):  (p+q)+r - (p+(q+r)) =', ax1)

# Axiom 2: commutativity
ax2 = (p + q - (q + p)).expand()
print('Axiom 2 (commutativity):  p+q - (q+p) =', ax2)

# Axiom 3: zero element. The zero polynomial is just 0.
ax3 = (p + 0 - p).expand()
print('Axiom 3 (zero):           p + 0 - p =', ax3)

# Axiom 4: additive inverse.
ax4 = (p + (-p)).expand()
print('Axiom 4 (inverse):        p + (-p) =', ax4)

# Axiom 5: scalar associativity
ax5 = (c*(d*p) - (c*d)*p).expand()
print('Axiom 5 (compatibility):  c(dp) - (cd)p =', ax5)

# Axiom 6: identity scalar
ax6 = (1*p - p).expand()
print('Axiom 6 (identity):       1·p - p =', ax6)

# Axiom 7: distributive over vector sum
ax7 = (c*(p + q) - (c*p + c*q)).expand()
print('Axiom 7 (dist. vec sum):  c(p+q) - (cp+cq) =', ax7)

# Axiom 8: distributive over scalar sum
ax8 = ((c + d)*p - (c*p + d*p)).expand()
print('Axiom 8 (dist. scal sum): (c+d)·p - (cp+dp) =', ax8)

All eight residuals are zero. `P₂` is a vector space.

The same eight verifications, with appropriate generic objects, certify any of the standard examples (`Pₙ`, `M_{m×n}`, `F(S, ℝ)`).


## 2. Polynomials in Sage — coefficient bridge

`PolynomialRing(QQ)` gives us exact-arithmetic polynomials. The `.list()` method extracts the coefficient vector in ascending order, which is the bridge from the abstract space `Pₙ` to the concrete ℝⁿ⁺¹.

In [ ]:
R.<x> = PolynomialRing(QQ)

def coeffs(p, deg):
    """Coefficient vector of p as an element of P_deg, padded with zeros."""
    lst = p.list()
    return vector(QQ, lst + [0]*(deg + 1 - len(lst)))

p = 3 + 2*x - x^2
q = 1 + x + x^2 - x^3
show('p =', p, '   coeffs in P_3 =', coeffs(p, 3))
show('q =', q, '   coeffs in P_3 =', coeffs(q, 3))
show('p + q =', p + q, '   coeffs =', coeffs(p + q, 3))
show('5 p =', 5*p, '   coeffs =', coeffs(5*p, 3))

## 3. Linear independence in `P₃` via coordinate stacking

Worked example E4: are `p₁ = 1 + x + x²`, `p₂ = x + x² + x³`, `p₃ = 1 + x³` independent?

In [ ]:
p1 = 1 + x + x^2
p2 = x + x^2 + x^3
p3 = 1 + x^3

M = column_matrix([coeffs(p1, 3), coeffs(p2, 3), coeffs(p3, 3)])
show('Coordinate matrix M (columns = p1, p2, p3 in basis (1,x,x^2,x^3)):')
show(M)
show('rank M =', M.rank(), '   (=3 means independent)')

## 4. Coordinates in a non-standard basis — Lagrange basis on `P₂`

The Lagrange basis `(ℓ₀, ℓ₁, ℓ₂)` for nodes `0, 1, 2` has the magical property that **coordinates of `p` in the Lagrange basis equal `(p(0), p(1), p(2))`**.

In [ ]:
nodes = [QQ(0), QQ(1), QQ(2)]

def lagrange(nodes, j):
    """Lagrange basis polynomial l_j for the given list of nodes."""
    p = R(1)
    for i, ni in enumerate(nodes):
        if i == j:
            continue
        p *= (x - ni) / (nodes[j] - ni)
    return p

ell0 = lagrange(nodes, 0)
ell1 = lagrange(nodes, 1)
ell2 = lagrange(nodes, 2)
show('l_0(x) =', ell0)
show('l_1(x) =', ell1)
show('l_2(x) =', ell2)

# Sanity: l_j(node_i) = delta_ij
for j, lj in enumerate([ell0, ell1, ell2]):
    show(f'l_{j} at nodes:', [lj.subs(x=n) for n in nodes])

In [ ]:
# Worked Example 10: p(x) = 3 + 2x - x^2
p = 3 + 2*x - x^2

# Coordinates in standard basis (1, x, x^2)
show('Standard-basis coordinates [p]_B =', coeffs(p, 2))

# Coordinates in Lagrange basis: just function values at the nodes
lag_coords = vector(QQ, [p.subs(x=n) for n in nodes])
show('Lagrange-basis coordinates [p]_L =', lag_coords)

# Verify: rebuild p as combination of Lagrange polys
rebuilt = sum(c * lj for c, lj in zip(lag_coords, [ell0, ell1, ell2]))
show('rebuilt =', rebuilt.expand(), '   (should match p)')

## 5. Matrix of the derivative `D : P₃ → P₂`

Bases: `𝓑 = (1, x, x², x³)` for `P₃`, `𝓒 = (1, x, x²)` for `P₂`. We build `[D]_{𝓒 ← 𝓑}` column by column — apply `D` to each basis vector of `P₃` and coordinatize in `𝓒`.

In [ ]:
basis_B = [R(1), x, x^2, x^3]
basis_C = [R(1), x, x^2]

D_columns = []
for v in basis_B:
    Dv = v.derivative(x)
    D_columns.append(coeffs(Dv, 2))

D_matrix = column_matrix(D_columns)
show('[D]_{C <- B} ='); show(D_matrix)

In [ ]:
# Sanity check: differentiate p = 3 x^3 - x + 2 by matrix multiplication
p = 3*x^3 - x + 2
p_coords = coeffs(p, 3)
Dp_coords = D_matrix * p_coords

# Decode back to a polynomial
Dp_from_matrix = sum(c * v for c, v in zip(Dp_coords, basis_C))
show('p =', p)
show('p\' (by Sage) =', p.derivative(x))
show('p\' (via matrix) =', Dp_from_matrix)
show('Match?', Dp_from_matrix == p.derivative(x))

## 6. Composition `D ∘ M_x` — matrix product

`M_x : P₂ → P₃` multiplies by `x`. Build its matrix and verify the composition `D ∘ M_x : P₂ → P₂` matches matrix multiplication `[D] · [M_x]`.

In [ ]:
basis_B2 = [R(1), x, x^2]            # basis of P_2 (domain of M_x)
basis_B3 = [R(1), x, x^2, x^3]       # basis of P_3 (codomain of M_x = domain of D)

# Matrix of M_x : P_2 -> P_3
Mx_columns = [coeffs(x * v, 3) for v in basis_B2]
Mx_matrix = column_matrix(Mx_columns)
show('[M_x] : P_2 -> P_3 ='); show(Mx_matrix)

# D : P_3 -> P_2 already built above as `D_matrix`.
# Composition D ∘ M_x : P_2 -> P_2 has matrix D_matrix * Mx_matrix
DMx_via_product = D_matrix * Mx_matrix
show('[D] · [M_x] (composition matrix on P_2) ='); show(DMx_via_product)

# Direct: D(M_x(p)) = D(x · p) = p + x p'  -- by product rule.
# Build matrix directly.
DMx_columns_direct = []
for v in basis_B2:
    DMx_v = (x * v).derivative(x)
    DMx_columns_direct.append(coeffs(DMx_v, 2))
DMx_direct = column_matrix(DMx_columns_direct)
show('Direct [D ∘ M_x] ='); show(DMx_direct)
show('Match?', DMx_via_product == DMx_direct)

## 7. Matrix of `T(p) = p + p'` on `P₂`, and similarity

Worked example E9. Build `[T]_𝓑` in `𝓑 = (1, x, x²)` and `[T]_{𝓑'}` in `𝓑' = (1, 1+x, 1+x+x²)`, then verify the **similarity formula**.

In [ ]:
B_polys  = [R(1), x, x^2]
Bp_polys = [R(1), 1 + x, 1 + x + x^2]

# Matrix of T(p) = p + p' in basis B
T_B_columns = [coeffs(v + v.derivative(x), 2) for v in B_polys]
T_B = column_matrix(T_B_columns)
show('[T]_B ='); show(T_B)

# P_inv = [id]_{B <- B'}: columns are B'-vectors expressed in B
P_inv = column_matrix([coeffs(p, 2) for p in Bp_polys])
P = P_inv.inverse()
show('P (B -> B\') ='); show(P)

# [T]_{B'} via similarity
T_Bp_via_similarity = P * T_B * P_inv
show('[T]_{B\'} via similarity ='); show(T_Bp_via_similarity)

# [T]_{B'} directly
def coords_in_Bp(p):
    """Find a, b, c such that p = a*1 + b*(1+x) + c*(1+x+x^2)."""
    return P_inv.solve_right(coeffs(p, 2))

T_Bp_direct_cols = [coords_in_Bp(v + v.derivative(x)) for v in Bp_polys]
T_Bp_direct = column_matrix(T_Bp_direct_cols)
show('[T]_{B\'} directly ='); show(T_Bp_direct)
show('Match?', T_Bp_via_similarity == T_Bp_direct)

## 8. Isomorphisms

`P₃ ≅ ℝ⁴` and `M_{2×2} ≅ ℝ⁴`. Both have dimension 4, so by the classification theorem they are isomorphic to each other and to `ℝ⁴`.

Concretely:

In [ ]:
# P_3 -> R^4
p = 5 - 3*x + 2*x^2 + 7*x^3
phi_p = coeffs(p, 3)
show('p =', p, '   phi(p) =', phi_p)

# M_{2x2} -> R^4 via row-major flatten
A = matrix(QQ, [[5, -3], [2, 7]])
show('A ='); show(A)
show('vec(A) =', vector(QQ, A.list()))

# So under the isomorphism, p and A map to the same vector. They live in
# different abstract spaces but share linear-algebraic structure.
print('phi(p) == vec(A)?', vector(QQ, A.list()) == phi_p)

## 9. Trace-zero `2×2` matrices

Worked example E3 / Exercise 7(b). Find a basis of the subspace `W = {A ∈ M_{2×2} : tr A = 0}` and verify dim 3.

In [ ]:
# Express trace as a linear functional in the standard basis (E11, E12, E21, E22).
# Coordinates of A: (a, b, c, d) for A = [[a, b], [c, d]]. Trace = a + d.
# So the kernel of the trace functional is the subspace defined by a + d = 0.

# Trace functional as a 1x4 row matrix
tr_matrix = matrix(QQ, [[1, 0, 0, 1]])
show('[tr] in standard basis ='); show(tr_matrix)

ker_tr = tr_matrix.right_kernel()
show('ker(tr) ='); show(ker_tr)
show('dim(ker tr) =', ker_tr.dimension())

# Convert basis vectors back to matrices for clarity
print('Basis matrices:')
for v in ker_tr.basis():
    M2 = matrix(QQ, 2, 2, list(v))
    show(M2, '   trace =', M2.trace())

---

## 10. Exercises (with solutions)

Each exercise lets you cross-check a hand calculation symbolically.

### E1 — Verify a polynomial subspace symbolically

Let `W = {p ∈ P₃ : p(0) + p(1) = 0}`. Verify all three subspace axioms symbolically. Find a basis of `W` and `dim W`.

### E2 — Matrix of "evaluate at three points"

Let `T : P₂ → ℝ³` be `T(p) = (p(-1), p(0), p(1))`. Build the matrix in standard bases. Find its kernel (the polynomials of degree ≤ 2 vanishing at all three points) and image. Use the dimensions to check rank–nullity.

### E3 — Pascal's "shift" matrix

Recompute the matrix of `T(p)(x) = p(x + 1)` on `P₃` in basis `(1, x, x², x³)`. (Worked example E7 did `P₂`; now do `P₃`.) Verify by computing `T(2 + 5x - x² + 3x³)` two ways.

### E4 — Change of basis

Let `T: P₂ → P₂` be `T(p)(x) = p(x + 1)` (the shift). Compute `[T]_{𝓑}` in `𝓑 = (1, x, x²)` and `[T]_{𝓒}` in the Lagrange basis `𝓒 = (ℓ₀, ℓ₁, ℓ₂)` for nodes `0, 1, 2`. Verify they're similar.

Note: in the Lagrange basis, `T(ℓⱼ)(x) = ℓⱼ(x+1)`, and you can read off coordinates as values at the nodes.

### E5 — Symmetric `2×2` matrices as a vector space

Let `S = {A ∈ M_{2×2} : Aᵀ = A}`. Find a basis (three matrices) and verify `dim S = 3`. Check that the antisymmetric part `K = {A ∈ M_{2×2} : Aᵀ = -A}` has dimension 1, so `dim S + dim K = 4 = dim M_{2×2}` (this is the orthogonal direct sum decomposition `M_{2×2} = S ⊕ K`, a preview of Ch 7).

---

### Solutions

### A1

In [ ]:
# W = {p in P_3 : p(0) + p(1) = 0}
# eval0 + eval1 is linear, so W is a kernel and hence a subspace.
# Symbolic verification: parametrize W and substitute.
a1_, a2_, a3_, b1_, b2_, b3_, c_ = var('a1_ a2_ a3_ b1_ b2_ b3_ c_')
t = var('t')

# A general p in P_3 with p(0) + p(1) = 0
# Constraint: a0 + (a0 + a1 + a2 + a3) = 2*a0 + a1 + a2 + a3 = 0
# Solve for a0: a0 = -(a1 + a2 + a3) / 2
def in_W(a1, a2, a3):
    a0 = -(a1 + a2 + a3) / 2
    return a0 + a1*t + a2*t^2 + a3*t^3

# Axiom 1
zero_in = in_W(0, 0, 0)
print('0 in W?  in_W(0,0,0) =', zero_in)

# Axiom 2: sum of two W elements is in W
p1 = in_W(a1_, a2_, a3_)
p2 = in_W(b1_, b2_, b3_)
sum_p = p1 + p2
sum_constraint = (sum_p.subs({t: 0}) + sum_p.subs({t: 1})).expand()
print('(p+q)(0) + (p+q)(1) =', sum_constraint)

# Axiom 3: scalar multiple
cp = c_ * p1
scalar_constraint = (cp.subs({t: 0}) + cp.subs({t: 1})).expand()
print('(c·p)(0) + (c·p)(1) =', scalar_constraint)

# Basis: Three free parameters (a1, a2, a3). Read off basis polynomials by setting
# one parameter = 1 and the others = 0.
print()
print('Basis polynomials (set one of a1, a2, a3 to 1):')
print('  q1 = ', in_W(1, 0, 0).expand())
print('  q2 = ', in_W(0, 1, 0).expand())
print('  q3 = ', in_W(0, 0, 1).expand())
print()
print('dim W = 3  (matches rank-nullity: dim P_3 - dim image = 4 - 1)')

### A2

In [ ]:
# T : P_2 -> R^3, T(p) = (p(-1), p(0), p(1))
basis_P2 = [R(1), x, x^2]
T_columns = []
for v in basis_P2:
    T_columns.append(vector(QQ, [v.subs(x=-1), v.subs(x=0), v.subs(x=1)]))
T_matrix = column_matrix(T_columns)
show('Matrix of T in standard bases:'); show(T_matrix)
show('rank T =', T_matrix.rank())
show('nullity T =', T_matrix.right_kernel().dimension())
show('rank + nullity =', T_matrix.rank() + T_matrix.right_kernel().dimension(),
     '   dim(P_2) = 3')

# T is onto (rank 3 = dim R^3), kernel is trivial.

### A3

In [ ]:
# Pascal's shift on P_3 in basis (1, x, x^2, x^3)
basis_P3 = [R(1), x, x^2, x^3]
shift_columns = []
for v in basis_P3:
    shifted = v.subs(x=x+1).expand()
    shift_columns.append(coeffs(shifted, 3))
shift_matrix = column_matrix(shift_columns)
show('Pascal shift matrix on P_3:'); show(shift_matrix)

# Sanity check on p = 2 + 5x - x^2 + 3x^3
p = 2 + 5*x - x^2 + 3*x^3
p_coords = coeffs(p, 3)
shifted_coords = shift_matrix * p_coords
shifted_from_matrix = sum(c * v for c, v in zip(shifted_coords, basis_P3))
shifted_direct = p.subs(x=x+1).expand()

show('p(x+1) (matrix) =', shifted_from_matrix)
show('p(x+1) (Sage)   =', shifted_direct)
show('Match?', shifted_from_matrix == shifted_direct)

### A4

In [ ]:
# T(p) = p(x+1) on P_2.  Standard basis B = (1, x, x^2);  Lagrange basis C at nodes 0, 1, 2.
nodes = [QQ(0), QQ(1), QQ(2)]
ell = [lagrange(nodes, j) for j in range(3)]

# Matrix in B
basis_P2 = [R(1), x, x^2]
T_B_cols = [coeffs((v.subs(x=x+1)).expand(), 2) for v in basis_P2]
T_B = column_matrix(T_B_cols)
show('[T]_B ='); show(T_B)

# Matrix in C: T(l_j) is l_j(x+1); its Lagrange coordinates are values at 0,1,2.
T_C_cols = []
for lj in ell:
    shifted = (lj.subs(x=x+1)).expand()
    coords_in_C = vector(QQ, [shifted.subs(x=n) for n in nodes])
    T_C_cols.append(coords_in_C)
T_C = column_matrix(T_C_cols)
show('[T]_C ='); show(T_C)

# Change-of-basis matrix from C to B
# P_BC := [id]_{B <- C}: columns are C-basis vectors expressed in B (i.e. coefficients of l_j)
P_BC = column_matrix([coeffs(lj, 2) for lj in ell])
P_CB = P_BC.inverse()

T_C_via_similarity = P_CB * T_B * P_BC
show('[T]_C via similarity ='); show(T_C_via_similarity)
show('Match?', T_C == T_C_via_similarity)

# trace and determinant should agree
show('tr [T]_B =', T_B.trace(), '  tr [T]_C =', T_C.trace())
show('det [T]_B =', T_B.det(),  '  det [T]_C =', T_C.det())

### A5

In [ ]:
# Symmetric 2x2 matrices: A = [[a, b], [b, c]]
# Basis: E11, (E12 + E21), E22  -- three matrices
S1 = matrix(QQ, [[1, 0], [0, 0]])
S2 = matrix(QQ, [[0, 1], [1, 0]])
S3 = matrix(QQ, [[0, 0], [0, 1]])

# Stack as 4-vectors (row-major) and check independence
basis_S = column_matrix([vector(QQ, S.list()) for S in [S1, S2, S3]])
show('Basis matrix for symmetric (rows = E11, E12, E21, E22 entries):'); show(basis_S)
show('rank =', basis_S.rank(), '   dim S = 3')

# Antisymmetric: A = [[0, b], [-b, 0]]
K1 = matrix(QQ, [[0, 1], [-1, 0]])
basis_K = column_matrix([vector(QQ, K1.list())])
show('Basis matrix for antisymmetric:'); show(basis_K)
show('dim K = 1')

# Check: dim S + dim K = 3 + 1 = 4 = dim M_{2x2}
show('dim S + dim K =', basis_S.rank() + basis_K.rank(), '   dim M_{2x2} = 4')

---

## Where to next

- **Chapter 7 (Sage):** Inner products and orthogonality. We'll add the extra structure that lets us measure length, angle, and orthogonal projections — and turn any basis into an orthonormal one via Gram–Schmidt (= QR factorization).
- **Chapter 9 (Sage):** Eigenvalues and diagonalization. The Ch 6 machinery shines here: diagonalization is exactly the question *"in which basis is `[T]_𝓑` diagonal?"*. We'll find such bases (when they exist) by computing eigenvectors.
